# Eksperimen 11: Direct Multi-Step Forecasting (DMSF)
**Strategi Kunci: Restrukturisasi Dataset & Anchor Lags**

Eksperimen ini membongkar hambatan model murni eksogen (cuaca) yang mentok di skor 1.65. Kita akan menginjeksikan kembali memori autokorelasi (TMA masa lalu) ke dalam model, tetapi dengan cara **Direct Forecasting**, BUKAN *Recursive*.

Data latih direstrukturisasi secara radikal:
1. Kita memotong data latih di ratusan titik "Anchor" (jangkar waktu) di masa lalu.
2. Di setiap jangkar, kita merekam kondisi TMA saat itu (`tma_anchor`).
3. Model dilatih untuk memprediksi TMA hingga 43 hari ke depan dari jangkar tersebut, berbekal cuaca masa depan dan jarak waktu (`steps_ahead`).
4. Hasilnya: Model akan otomatis belajar untuk sangat mempercayai `tma_anchor` di jam-jam pertama, dan perlahan beralih ke sinyal cuaca saat memprediksi masa depan yang jauh.

In [1]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
import optuna
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Pemuatan Data

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

overall_cutoff = train['datetime'].max()
print("Batas Akhir Train:", overall_cutoff)
print("Awal Test        :", test['datetime'].min())

Batas Akhir Train: 2025-09-18 18:00:00
Awal Test        : 2025-09-19 06:00:00


## 2. Preprocessing & Agregasi Cuaca (Global)

In [3]:
env_data = env_data.sort_values(['nama_pos', 'datetime'])

macro_cols = ['nino_34', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'rmm1', 'rmm2']
dynamic_cols = ['surface_pressure_hpa', 'pressure_msl_hpa', 'soil_moisture_0_7cm',
                'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']

for c in macro_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].ffill().bfill()

for c in dynamic_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].apply(
        lambda x: x.interpolate(method='linear').bfill().ffill()
    ).reset_index(level=0, drop=True)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
coords['spatial_cluster'] = kmeans.fit_predict(coords[['latitude', 'longitude']])

def aggregate_env_data(df):
    agg_funcs = {col: 'mean' for col in df.columns if col not in ['nama_pos', 'landcover_name', 'datetime']}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    df_indexed = df.set_index('datetime')
    return df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()

env_agg = aggregate_env_data(env_data)
env_agg = pd.merge(env_agg, coords, on='nama_pos', how='left')

### 2.1 Feature Engineering pada Matriks Cuaca

In [4]:
env_agg['month'] = env_agg['datetime'].dt.month
env_agg['hour'] = env_agg['datetime'].dt.hour
env_agg['sin_hour'] = np.sin(2 * np.pi * env_agg['hour'] / 24)
env_agg['cos_hour'] = np.cos(2 * np.pi * env_agg['hour'] / 24)
env_agg['sin_month'] = np.sin(2 * np.pi * env_agg['month'] / 12)
env_agg['cos_month'] = np.cos(2 * np.pi * env_agg['month'] / 12)

env_agg['runoff_factor'] = env_agg['rainfall_mm'] * env_agg['soil_moisture_0_7cm']
env_agg['pressure_drop'] = env_agg.groupby('nama_pos')['surface_pressure_hpa'].diff(1).fillna(0)

windows = [4, 8, 24, 56]
for w in windows:
    env_agg[f'rainfall_roll_{w}'] = env_agg.groupby('nama_pos')['rainfall_mm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).sum()
    )
    env_agg[f'soil_roll_{w}'] = env_agg.groupby('nama_pos')['soil_moisture_0_7cm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )

le = LabelEncoder()
env_agg['nama_pos_encoded'] = le.fit_transform(env_agg['nama_pos'])

## 3. Station Normalization Profile (Murni dari Data Latih)

In [5]:
station_profile = train.groupby('nama_pos')['tma_mdpl'].agg(
    tma_mean='mean',
    tma_std='std',
    tma_p25=lambda x: x.quantile(0.25),
    tma_p75=lambda x: x.quantile(0.75)
).reset_index()
station_profile['tma_std'] = station_profile['tma_std'].fillna(1.0)

## 4. Dataset Restructuring: Direct Multi-Step
Proses krusial! Kita membuat *training set* yang merepresentasikan kondisi prediksi sebenarnya.

In [6]:
max_test_horizon = int((test['datetime'].max() - overall_cutoff) / pd.Timedelta('3H'))
print(f"Test Horizon Maximum: {max_test_horizon} steps ({max_test_horizon * 3 / 24:.1f} hari)")

start_date = train['datetime'].min() + pd.Timedelta('30D')
end_date = overall_cutoff - pd.Timedelta(f'{max_test_horizon * 3}H')
anchor_dates = pd.date_range(start=start_date, end=end_date, freq='20D')
print(f"Total Anchor Points di-sampling: {len(anchor_dates)}")

train_indexed = train.set_index(['nama_pos', 'datetime'])['tma_mdpl']

dfs = []
for anchor in anchor_dates:
    anchor_data = []
    for pos in train['nama_pos'].unique():
        if (pos, anchor) in train_indexed.index:
            tma_0 = train_indexed.loc[(pos, anchor)]
            tma_minus_1d = train_indexed.loc[(pos, anchor - pd.Timedelta('1D'))] if (pos, anchor - pd.Timedelta('1D')) in train_indexed.index else tma_0
            tma_minus_3d = train_indexed.loc[(pos, anchor - pd.Timedelta('3D'))] if (pos, anchor - pd.Timedelta('3D')) in train_indexed.index else tma_0
            
            horizon_times = pd.date_range(start=anchor + pd.Timedelta('3H'), periods=max_test_horizon, freq='3H')
            for h, h_time in enumerate(horizon_times):
                if (pos, h_time) in train_indexed.index:
                    tma_target = train_indexed.loc[(pos, h_time)]
                    anchor_data.append({
                        'nama_pos': pos,
                        'datetime': h_time,
                        'anchor_time': anchor,
                        'steps_ahead': h + 1,
                        'tma_anchor_0': tma_0,
                        'tma_anchor_1d': tma_minus_1d,
                        'tma_anchor_3d': tma_minus_3d,
                        'tma_target': tma_target
                    })
    if anchor_data:
        dfs.append(pd.DataFrame(anchor_data))
        
df_dmsf = pd.concat(dfs, ignore_index=True)
print(f"Restrukturisasi Train selesai. Dimensi baru: {df_dmsf.shape}")

Test Horizon Maximum: 1936 steps (242.0 hari)
Total Anchor Points di-sampling: 36
Restrukturisasi Train selesai. Dimensi baru: (725116, 8)


### 4.1 Menggabungkan Data Cuaca ke Matriks DMSF

In [7]:
df_dmsf = pd.merge(df_dmsf, env_agg, on=['nama_pos', 'datetime'], how='left')
df_dmsf = pd.merge(df_dmsf, station_profile, on='nama_pos', how='left')

df_dmsf['tma_anchor_0_norm'] = (df_dmsf['tma_anchor_0'] - df_dmsf['tma_mean']) / df_dmsf['tma_std']
df_dmsf['tma_anchor_1d_norm'] = (df_dmsf['tma_anchor_1d'] - df_dmsf['tma_mean']) / df_dmsf['tma_std']
df_dmsf['tma_anchor_3d_norm'] = (df_dmsf['tma_anchor_3d'] - df_dmsf['tma_mean']) / df_dmsf['tma_std']

df_dmsf['tma_target_norm'] = (df_dmsf['tma_target'] - df_dmsf['tma_mean']) / df_dmsf['tma_std']

df_dmsf['steps_x_sin_month'] = df_dmsf['steps_ahead'] * df_dmsf['sin_month']
df_dmsf['steps_x_cos_month'] = df_dmsf['steps_ahead'] * df_dmsf['cos_month']

print("Preview fitur pada baris sampel:")
print(df_dmsf[['steps_ahead', 'tma_anchor_0_norm', 'rainfall_mm', 'tma_target_norm']].head())

Preview fitur pada baris sampel:
   steps_ahead  tma_anchor_0_norm  rainfall_mm  tma_target_norm
0            2           0.418676          2.0         0.669730
1            4           0.418676          0.5         0.877184
2            8           0.418676          0.2         0.472506
3           10           0.418676          1.4         0.472506
4           12           0.418676          0.0         0.877184


## 5. Persiapan Test Set untuk Inferensi
Test set hanya memiliki 1 titik *anchor* mutlak, yaitu batas akhir data train (18 Sept 2025).

In [8]:
test_anchors = []
for pos in test['nama_pos'].unique():
    tma_0 = train_indexed.loc[(pos, overall_cutoff)] if (pos, overall_cutoff) in train_indexed.index else global_mean
    tma_minus_1d = train_indexed.loc[(pos, overall_cutoff - pd.Timedelta('1D'))] if (pos, overall_cutoff - pd.Timedelta('1D')) in train_indexed.index else tma_0
    tma_minus_3d = train_indexed.loc[(pos, overall_cutoff - pd.Timedelta('3D'))] if (pos, overall_cutoff - pd.Timedelta('3D')) in train_indexed.index else tma_0
    
    pos_test = test[test['nama_pos'] == pos].copy()
    for _, row in pos_test.iterrows():
        test_anchors.append({
            'id': row['id'],
            'nama_pos': pos,
            'datetime': row['datetime'],
            'steps_ahead': int((row['datetime'] - overall_cutoff) / pd.Timedelta('3H')),
            'tma_anchor_0': tma_0,
            'tma_anchor_1d': tma_minus_1d,
            'tma_anchor_3d': tma_minus_3d
        })

df_test_dmsf = pd.DataFrame(test_anchors)
df_test_dmsf = pd.merge(df_test_dmsf, env_agg, on=['nama_pos', 'datetime'], how='left')
df_test_dmsf = pd.merge(df_test_dmsf, station_profile, on='nama_pos', how='left')

df_test_dmsf['tma_anchor_0_norm'] = (df_test_dmsf['tma_anchor_0'] - df_test_dmsf['tma_mean']) / df_test_dmsf['tma_std']
df_test_dmsf['tma_anchor_1d_norm'] = (df_test_dmsf['tma_anchor_1d'] - df_test_dmsf['tma_mean']) / df_test_dmsf['tma_std']
df_test_dmsf['tma_anchor_3d_norm'] = (df_test_dmsf['tma_anchor_3d'] - df_test_dmsf['tma_mean']) / df_test_dmsf['tma_std']

df_test_dmsf['steps_x_sin_month'] = df_test_dmsf['steps_ahead'] * df_test_dmsf['sin_month']
df_test_dmsf['steps_x_cos_month'] = df_test_dmsf['steps_ahead'] * df_test_dmsf['cos_month']

## 6. Training Setup
Kita akan memvalidasi model menggunakan GroupKFold berdasarkan `anchor_time` untuk memastikan tidak ada *leakage* temporal antar-fold.

In [9]:
drop_cols = ['datetime', 'nama_pos', 'anchor_time', 'tma_target', 'tma_target_norm', 'id', 'landcover_name']
features = [c for c in df_dmsf.columns if c not in drop_cols]
target = 'tma_target_norm'

X_full = df_dmsf[features]
y_full = df_dmsf[target]
groups = df_dmsf['anchor_time']
X_test = df_test_dmsf[features]

from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)

print(f"Total fitur: {len(features)}")
print(features)

Total fitur: 57
['steps_ahead', 'tma_anchor_0', 'tma_anchor_1d', 'tma_anchor_3d', 'rainfall_mm', 'humidity_pct', 'wind_direction_deg', 'dew_point_c', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'rainfall_openmeteo_mm', 'rainfall_max_24h_mm', 'solar_radiation_mj_m2', 'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa', 'built_surface_m2', 'landcover_class', 'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'nino_34', 'latitude', 'longitude', 'spatial_cluster', 'month', 'hour', 'sin_hour', 'cos_hour', 'sin_month', 'cos_month', 'runoff_factor', 'pressure_drop', 'rainfall_roll_4', 'soil_roll_4', 'rainfall_roll_8', 'soil_roll_8', 'rainfall_roll_24', 'soil_roll_24', 'rainfall_roll_56', 'soil_roll_56', 'nama_pos_encoded', 'tma_mean', 'tma_std', 'tma_p25', 'tma_p75', 'tma_anchor_0_norm', 'tma_anchor_1d_norm', 'tma_anchor_3d_norm', 'steps_x_sin_month', 'steps_x_cos_month']


### 6.1 Optuna Tuning: Max Power LGBM
Mencari konfigurasi arsitektur terbaik untuk model baru kita.

In [10]:
def objective_lgb(trial):
    params = {
        'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 500),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'random_state': 42,
        'verbose': -1
    }
    
    # Subsample Optuna untuk kecepatan
    optuna_idx = np.random.choice(len(X_full), size=int(len(X_full)*0.3), replace=False)
    X_sub = X_full.iloc[optuna_idx]
    y_sub = y_full.iloc[optuna_idx]
    g_sub = groups.iloc[optuna_idx]
    
    scores = []
    for train_idx, val_idx in gkf.split(X_sub, y_sub, g_sub):
        X_tr, X_va = X_sub.iloc[train_idx], X_sub.iloc[val_idx]
        y_tr, y_va = y_sub.iloc[train_idx], y_sub.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
        scores.append(mean_squared_error(y_va, m.predict(X_va)))
    return np.mean(scores)

print("Memulai Optuna Tuning DMSF (30 trials)...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=30)
best_lgb = study_lgb.best_params
best_lgb.update({'n_estimators': 3000, 'random_state': 42, 'verbose': -1})
print(f"\nBest LightGBM Params: {best_lgb}")

Memulai Optuna Tuning DMSF (30 trials)...

Best LightGBM Params: {'learning_rate': 0.059133740912560566, 'num_leaves': 151, 'max_depth': 12, 'subsample': 0.7772134620928237, 'colsample_bytree': 0.6035222933877891, 'min_child_samples': 50, 'reg_alpha': 0.6809612647763237, 'reg_lambda': 0.2516902723933659, 'n_estimators': 3000, 'random_state': 42, 'verbose': -1}


## 7. K-Fold Training & Prediction
Melatih final model pada 100% data ter-restrukturisasi secara utuh.

In [11]:
test_preds_lgb = np.zeros(len(X_test))
cv_rmse_scores = []

print("Memulai K-Fold Final Training...")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_full, y_full, groups)):
    X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]

    val_mean = df_dmsf.iloc[val_idx]['tma_mean'].values
    val_std = df_dmsf.iloc[val_idx]['tma_std'].values

    m_lgb = lgb.LGBMRegressor(**best_lgb)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])

    p_norm = m_lgb.predict(X_va)
    p_abs = (p_norm * val_std) + val_mean
    y_abs = (y_va.values * val_std) + val_mean

    fold_rmse = np.sqrt(mean_squared_error(y_abs, p_abs))
    cv_rmse_scores.append(fold_rmse)
    print(f"Fold {fold+1} RMSE (abs): {fold_rmse:.4f} | Trees Matang: {m_lgb.best_iteration_}")

    test_preds_lgb += m_lgb.predict(X_test) / gkf.n_splits

print(f"\nRata-rata K-Fold RMSE (DMSF): {np.mean(cv_rmse_scores):.4f}")

Memulai K-Fold Final Training...
Fold 1 RMSE (abs): 0.4223 | Trees Matang: 3000
Fold 2 RMSE (abs): 0.3627 | Trees Matang: 3000
Fold 3 RMSE (abs): 0.3367 | Trees Matang: 3000
Fold 4 RMSE (abs): 0.2705 | Trees Matang: 3000
Fold 5 RMSE (abs): 0.1726 | Trees Matang: 3000

Rata-rata K-Fold RMSE (DMSF): 0.3129


## 8. Denormalisasi & Output Submisi

In [12]:
test_mean = df_test_dmsf['tma_mean'].values
test_std = df_test_dmsf['tma_std'].values
final_tma = (test_preds_lgb * test_std) + test_mean

df_test_dmsf['tma_mdpl'] = final_tma
submission = df_test_dmsf[['id', 'tma_mdpl']]

if not os.path.exists('../submissions'):
    os.makedirs('../submissions')

submission.to_csv('../submissions/submission.csv', index=False)
print("Eksperimen 11 selesai. File submission.csv tersimpan.")
print(f"Rentang prediksi TMA: {final_tma.min():.2f} - {final_tma.max():.2f}")

Eksperimen 11 selesai. File submission.csv tersimpan.
Rentang prediksi TMA: 0.71 - 144.79
